In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

print("HF cache exists:", os.path.exists("/root/.cache/huggingface"))
print("Transformers cache exists:", os.path.exists("/root/.cache/huggingface/hub"))

if os.path.exists("/root/.cache/huggingface/hub"):
    print("\nCached models:")
    for x in os.listdir("/root/.cache/huggingface/hub"):
        print(x)

HF cache exists: True
Transformers cache exists: True

Cached models:
.locks
CACHEDIR.TAG
models--facebook--nllb-200-distilled-600M


In [ ]:
import requests

try:
    response = requests.get("https://huggingface.co", timeout=10)
    print("Internet connection: True")
    print("Status code:", response.status_code)
except Exception as e:
    print("Internet connection: False")
    print("Error:", e)

Internet connection: True
Status code: 200


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/nllb-200-distilled-600M"

print("Loading NLLB model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
)

model = model.to("cuda")

print("Model loaded successfully!")
print("Device:", next(model.parameters()).device)

Loading NLLB model...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0


In [ ]:
source_text = "Mom, let's go for a movie tomorrow."

tokenizer.src_lang = "eng_Latn"

inputs = tokenizer(
    source_text,
    return_tensors="pt"
).to("cuda")

generated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("guj_Gujr"),
    max_length=200
)

translation = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)[0]

print("English:")
print(source_text)

print("\nGujarati:")
print(translation)

English:
Mom, let's go for a movie tomorrow.

Gujarati:
મમ્મી, ચાલો આપણે કાલે ફિલ્મ પર જઈએ.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Language mapping
LANG_CODES = {
    "Gujarati": "guj_Gujr",
    "Hindi": "hin_Deva",
    "Kannada": "kan_Knda",
    "Malayalam": "mal_Mlym",
    "Marathi": "mar_Deva",
    "Odia": "ory_Orya"
}

def translate_batch(texts, target_lang, batch_size=16, max_length=256):

    translations = []

    target_token_id = tokenizer.convert_tokens_to_ids(target_lang)

    for i in tqdm(
        range(0, len(texts), batch_size),
        desc=f"Translating → {target_lang}"
    ):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to("cuda")

        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=target_token_id,
                max_length=max_length
            )

        batch_translations = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True
        )

        translations.extend(batch_translations)

    return translations

In [ ]:
test_sentence = "This is a test sentence for machine translation."

for language, code in LANG_CODES.items():

    result = translate_batch(
        [test_sentence],
        code,
        batch_size=1
    )

    print(f"\n{language}:")
    print(result[0])


Translating → guj_Gujr:   0%|          | 0/1 [00:00<?, ?it/s]


Gujarati:
આ મશીન અનુવાદ માટે એક પરીક્ષણ વાક્ય છે.


Translating → hin_Deva:   0%|          | 0/1 [00:00<?, ?it/s]


Hindi:
यह मशीन अनुवाद के लिए एक परीक्षण वाक्य है।


Translating → kan_Knda:   0%|          | 0/1 [00:00<?, ?it/s]

...


Malayalam:
ഇത് യന്ത്ര വിവർത്തനത്തിന് വേണ്ടിയുള്ള ഒരു പരീക്ഷണ വാക്യമാണ്.


Translating → mar_Deva:   0%|          | 0/1 [00:00<?, ?it/s]


Marathi:
मशीन अनुवादासाठी हा एक चाचणी वाक्य आहे.


Translating → ory_Orya:   0%|          | 0/1 [00:00<?, ?it/s]


Odia:
ଏହା ମେସିନ୍ ଅନୁବାଦ ପାଇଁ ଏକ ପରୀକ୍ଷା ବାକ୍ୟ ।


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

FLORES_DIR = PROJECT_ROOT / "datasets" / "FLORES200"
MT_OUTPUT_DIR = PROJECT_ROOT / "processed" / "mt_outputs" / "flores200"

MT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Mapping: file → target language
FLORES_FILES = {
    "Gujarati": "Gujarati_clean.xlsx",
    "Hindi": "Hindi_clean.xlsx",
    "Kannada": "Kannada_clean.xlsx",
    "Malayalam": "Malayalam_clean.xlsx",
    "Marathi": "Marathi_clean.xlsx",
    "Odia": "Odia_clean.xlsx"
}

all_flores = {}

for language, filename in FLORES_FILES.items():

    print("\n" + "=" * 60)
    print(f"Processing {language}")
    print("=" * 60)

    file_path = FLORES_DIR / filename

    df = pd.read_excel(file_path)

    # Identify source and reference columns
    source_col = "sentence_eng_Latn"
    reference_col = [
        col for col in df.columns
        if col != source_col
    ][0]

    print("Rows:", len(df))
    print("Source column:", source_col)
    print("Reference column:", reference_col)

    # Generate machine translations
    translations = translate_batch(
        df[source_col].astype(str).tolist(),
        LANG_CODES[language],
        batch_size=16
    )

    # Add hypothesis column
    df["translation"] = translations

    # Save
    output_file = MT_OUTPUT_DIR / f"{language}_mt.csv"
    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    all_flores[language] = df

    print(f"Saved: {output_file}")
    print("Translation examples:")
    print(df[[source_col, reference_col, "translation"]].head(2))


Processing Gujarati
Rows: 997
Source column: sentence_eng_Latn
Reference column: sentence_guj_Gujr


Translating → guj_Gujr:   0%|          | 0/63 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200/Gujarati_mt.csv
Translation examples:
                                   sentence_eng_Latn  \
0  On Monday, scientists from the Stanford Univer...   
1  Lead researchers say this may bring early dete...   

                                   sentence_guj_Gujr  \
0  સોમવારે, સ્ટેનફોર્ડ યુનિવર્સિટી સ્કૂલ ઓફ મેડિસ...   
1  અગ્રણી સંશોધનકર્તાઓ કહે છે કે, સમૃદ્ધ દેશોની સ...   

                                         translation  
0  સોમવારે, સ્ટેનફોર્ડ યુનિવર્સિટી સ્કૂલ ઓફ મેડિસ...  
1  અગ્રણી સંશોધકો કહે છે કે આ કેન્સર, ક્ષય રોગ, એ...  

Processing Hindi
Rows: 997
Source column: sentence_eng_Latn
Reference column: sentence_hin_Deva


Translating → hin_Deva:   0%|          | 0/63 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200/Hindi_mt.csv
Translation examples:
                                   sentence_eng_Latn  \
0  On Monday, scientists from the Stanford Univer...   
1  Lead researchers say this may bring early dete...   

                                   sentence_hin_Deva  \
0  सोमवार को, स्टैनफ़ोर्ड यूनिवर्सिटी स्कूल ऑफ़ म...   
1  शोधकर्ताओं ने कहा है कि यह अल्प आय वाले देशों ...   

                                         translation  
0  सोमवार को स्टैनफोर्ड विश्वविद्यालय के मेडिकल स...  
1  प्रमुख शोधकर्ताओं का कहना है कि इससे कम आय वाल...  

Processing Kannada
Rows: 997
Source column: sentence_eng_Latn
Reference column: sentence_kan_Knda


...

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200/Malayalam_mt.csv
Translation examples:
                                   sentence_eng_Latn  \
0  On Monday, scientists from the Stanford Univer...   
1  Lead researchers say this may bring early dete...   

                                   sentence_mal_Mlym  \
0  തിങ്കളാഴ്ച്ച, സ്റ്റാൻഫോർഡ് യൂണിവേഴ്‌സിറ്റി സ്‌...   
1  മുന്‍നിര ഗവേഷകര്‍ പറയുന്നത് ഇത് നേരത്തേതന്നെ അ...   

                                         translation  
0  തിങ്കളാഴ്ച സ്റ്റാൻഫോർഡ് യൂണിവേഴ്സിറ്റി മെഡിക്ക...  
1  മുഖ്യ ഗവേഷകർ പറയുന്നത്, ഇത് താഴ്ന്ന വരുമാനമുള്...  

Processing Marathi
Rows: 997
Source column: sentence_eng_Latn
Reference column: sentence_mar_Deva


Translating → mar_Deva:   0%|          | 0/63 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200/Marathi_mt.csv
Translation examples:
                                   sentence_eng_Latn  \
0  On Monday, scientists from the Stanford Univer...   
1  Lead researchers say this may bring early dete...   

                                   sentence_mar_Deva  \
0  स्टॅनफोर्ड युनिव्हर्सिटी स्कूल ऑफ मेडिसीनमधील ...   
1  सुप्रसिद्ध संशोधकांच्या मते, यामुळे अल्प उत्पन...   

                                         translation  
0  सोमवारी स्टॅन्फोर्ड विद्यापीठाच्या वैद्यकीय शा...  
1  याबाबतचे प्रमुख संशोधक म्हणतात की, यामुळे कमी ...  

Processing Odia
Rows: 997
Source column: sentence_eng_Latn
Reference column: sentence_ory_Orya


Translating → ory_Orya:   0%|          | 0/63 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200/Odia_mt.csv
Translation examples:
                                   sentence_eng_Latn  \
0  On Monday, scientists from the Stanford Univer...   
1  Lead researchers say this may bring early dete...   

                                   sentence_ory_Orya  \
0  ସୋମବାର ଦିନ, ଷ୍ଟାନଫୋର୍ଡ ୟୁନିଭରସିଟି ସ୍କୁଲ୍ ଅଫ୍ ମ...   
1  ଅଗ୍ରଣୀ ଅନୁସନ୍ଧାନକାରୀମାନେ କହନ୍ତି ଯେ ଏହା କମ ଆୟ କ...   

                                         translation  
0  ସୋମବାର ଦିନ ଷ୍ଟାନଫୋର୍ଡ ୟୁନିଭର୍ସିଟି ମେଡିକାଲ ସ୍କୁ...  
1  ଗବେଷକମାନେ କହିଛନ୍ତି ଯେ ଏହା ଦ୍ୱାରା କମ୍ ଆୟକାରୀ ଦେ...  


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

FLORES_DIR = PROJECT_ROOT / "datasets" / "FLORES200"
MT_OUTPUT_DIR = PROJECT_ROOT / "processed" / "mt_outputs" / "flores200_reverse"

MT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# FLORES language configuration
FLORES_LANGUAGES = {
    "Gujarati": {
        "file": "Gujarati_clean.xlsx",
        "source_col": "sentence_guj_Gujr",
        "lang_code": "guj_Gujr"
    },
    "Hindi": {
        "file": "Hindi_clean.xlsx",
        "source_col": "sentence_hin_Deva",
        "lang_code": "hin_Deva"
    },
    "Kannada": {
        "file": "Kannada_clean.xlsx",
        "source_col": "sentence_kan_Knda",
        "lang_code": "kan_Knda"
    },
    "Malayalam": {
        "file": "Malayalam_clean.xlsx",
        "source_col": "sentence_mal_Mlym",
        "lang_code": "mal_Mlym"
    },
    "Marathi": {
        "file": "Marathi_clean.xlsx",
        "source_col": "sentence_mar_Deva",
        "lang_code": "mar_Deva"
    },
    "Odia": {
        "file": "Odia_clean.xlsx",
        "source_col": "sentence_ory_Orya",
        "lang_code": "ory_Orya"
    }
}

for language, config in FLORES_LANGUAGES.items():

    print("\n" + "=" * 70)
    print(f"Processing {language} → English")
    print("=" * 70)

    file_path = FLORES_DIR / config["file"]

    df = pd.read_excel(file_path)

    source_col = config["source_col"]
    reference_col = "sentence_eng_Latn"

    print("Rows:", len(df))
    print("Source:", source_col)
    print("Reference:", reference_col)

    # Set NLLB source language
    tokenizer.src_lang = config["lang_code"]

    # Generate Indic → English translations
    translations = translate_batch(
        df[source_col].astype(str).tolist(),
        "eng_Latn",
        batch_size=16
    )

    # Create standardized output
    output_df = pd.DataFrame({
        "source": df[source_col].astype(str),
        "reference": df[reference_col].astype(str),
        "translation": translations,
        "source_language": language,
        "target_language": "English",
        "direction": f"{config['lang_code']}→eng_Latn",
        "dataset": "FLORES-200"
    })

    # Save
    output_file = MT_OUTPUT_DIR / f"{language}_to_English.csv"

    output_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\nSaved: {output_file}")

    print("\nSample:")
    print(
        output_df[
            ["source", "reference", "translation"]
        ].head(2).to_string(index=False)
    )


Processing Gujarati → English
Rows: 997
Source: sentence_guj_Gujr
Reference: sentence_eng_Latn


Translating → eng_Latn:   0%|          | 0/63 [00:00<?, ?it/s]


Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200_reverse/Gujarati_to_English.csv

Sample:
                                                                                                                                                                                                                                                                    source                                                                                                                                                                                                                                                                reference                                                                                                                                                                                                                                             translation
સોમવારે, સ્ટેનફોર્ડ યુનિવર્સિટી સ્કૂલ ઓફ મેડિસિનના વૈજ્ઞાનિકોએ એક નવા નિદાન સાધનની શોધની જાહેરાત કરી હતી જ

Translating → eng_Latn:   0%|          | 0/63 [00:00<?, ?it/s]


Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200_reverse/Hindi_to_English.csv

Sample:
                                                                                                                                                                                                                                                                                                     source                                                                                                                                                                                                                                                                reference                                                                                                                                                                                                                                              translation
सोमवार को, स्टैनफ़ोर्ड यूनिवर्सिटी स्कूल ऑफ़ मेडिसिन के वैज्ञानिकों ने एक न

...


Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200_reverse/Malayalam_to_English.csv

Sample:
                                                                                                                                                                                                                                                                                                                                                                  source                                                                                                                                                                                                                                                                reference                                                                                                                                                                                                                                                                    

Translating → eng_Latn:   0%|          | 0/63 [00:00<?, ?it/s]


Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200_reverse/Marathi_to_English.csv

Sample:
                                                                                                                                                                                                                                                                          source                                                                                                                                                                                                                                                                reference                                                                                                                                                                                                                                                     translation
स्टॅनफोर्ड युनिव्हर्सिटी स्कूल ऑफ मेडिसीनमधील शास्त्रज्ञांनी पेशींचे प्रकारानुसार विभाजन करणा

Translating → eng_Latn:   0%|          | 0/63 [00:00<?, ?it/s]


Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/flores200_reverse/Odia_to_English.csv

Sample:
                                                                                                                                                                                                                                                                                                              source                                                                                                                                                                                                                                                                reference                                                                                                                                                                                                                                                   translation
ସୋମବାର ଦିନ, ଷ୍ଟାନଫୋର୍ଡ ୟୁନିଭରସିଟି ସ୍କୁଲ୍ ଅଫ୍ ମେଡିସିନ୍ ର ବୈଜ୍ଞା

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

forward_dir = PROJECT_ROOT / "processed" / "mt_outputs" / "flores200"
reverse_dir = PROJECT_ROOT / "processed" / "mt_outputs" / "flores200_reverse"

print("FORWARD FILES")
print("-" * 50)

for file in sorted(forward_dir.glob("*.csv")):
    df = pd.read_csv(file)
    print(f"{file.name:30} {len(df)} rows")

print("\nREVERSE FILES")
print("-" * 50)

for file in sorted(reverse_dir.glob("*.csv")):
    df = pd.read_csv(file)
    print(f"{file.name:30} {len(df)} rows")

FORWARD FILES
--------------------------------------------------
Gujarati_mt.csv                997 rows
Hindi_mt.csv                   997 rows
Kannada_mt.csv                 997 rows
Malayalam_mt.csv               997 rows
Marathi_mt.csv                 997 rows
Odia_mt.csv                    997 rows

REVERSE FILES
--------------------------------------------------
Gujarati_to_English.csv        997 rows
Hindi_to_English.csv           997 rows
Kannada_to_English.csv         997 rows
Malayalam_to_English.csv       997 rows
Marathi_to_English.csv         997 rows
Odia_to_English.csv            997 rows


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")
IN22_DIR = PROJECT_ROOT / "datasets" / "IN22"

for subset in ["Conv", "Gen"]:

    subset_dir = IN22_DIR / subset

    print("\n" + "=" * 70)
    print(f"IN22 {subset}")
    print("=" * 70)

    files = sorted(subset_dir.iterdir())

    for file in files:
        if file.is_file():
            print(f"\nFILE: {file.name}")

            # Read first few lines
            try:
                with open(file, "r", encoding="utf-8") as f:
                    lines = [next(f).strip() for _ in range(3)]

                for i, line in enumerate(lines, 1):
                    print(f"  Line {i}: {line[:150]}")

            except Exception as e:
                print("  Error:", e)


IN22 Conv

FILE: test (1).eng_Latn
  Line 1: Mom, let's go for a movie tomorrow.
  Line 2: I don't have to go to school.
  Line 3: It is a holiday.

FILE: test (1).guj_Gujr
  Line 1: મમ્મી, ચાલો આપણે આવતીકાલે ફિલ્મ જોવા જઇએ.
  Line 2: મારે કાલે શાળાએ જવાનું નથી.
  Line 3: રજા છે.

FILE: test (1).hin_Deva
  Line 1: माँ, चलो कल एक फिल्म देखने चलते हैं।
  Line 2: मुझे कल स्कूल नहीं जाना है।
  Line 3: छुट्टी है।

FILE: test (1).mar_Deva
  Line 1: आई, उद्या आपण एखादा सिनेमा बघायला जाऊया.
  Line 2: मला शाळेत जायचं नाहीये.
  Line 3: उद्या सुट्टी आहे.

FILE: test.kan_Knda
  Line 1: ಅಮ್ಮಾ, ನಾಳೆ ನಾವು ಸಿನ್ಮಾ ನೋಡಕ್‌ ಹೋಗಣ.
  Line 2: ನಂಗೆ ಸ್ಕೂಲಿಲ್ಲ.
  Line 3: ರಜೆಯಿದೆ.

FILE: test.mal_Mlym
  Line 1: അമ്മേ, നാളെ നമുക്ക് ഒരു സിനിമയ്ക്ക് പോകാം.
  Line 2: എനിക്കു സ്കൂളില്ല.
  Line 3: അവധിയാ.

FILE: test.ory_Orya
  Line 1: ମା', ଚାଲ କାଲି ଗୋଟେ ସିନେମା ଦେଖିବାକୁ ଯିବା।
  Line 2: ମୋର ସ୍କୁଲ ଯିବାର ନାହିଁ।
  Line 3: ଛୁଟି ଅଛି।

IN22 Gen

FILE: test.eng_Latn
  Line 1: An appearance is a bunch of attributes related to

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

IN22_DIR = PROJECT_ROOT / "datasets" / "IN22"
OUTPUT_DIR = PROJECT_ROOT / "processed" / "mt_outputs" / "in22"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Language configuration
LANGUAGES = {
    "Gujarati": {
        "code": "guj_Gujr",
        "filename": {
            "Conv": "test (1).guj_Gujr",
            "Gen": "test.guj_Gujr"
        }
    },
    "Hindi": {
        "code": "hin_Deva",
        "filename": {
            "Conv": "test (1).hin_Deva",
            "Gen": "test.hin_Deva"
        }
    },
    "Kannada": {
        "code": "kan_Knda",
        "filename": {
            "Conv": "test.kan_Knda",
            "Gen": "test.kan_Knda"
        }
    },
    "Malayalam": {
        "code": "mal_Mlym",
        "filename": {
            "Conv": "test.mal_Mlym",
            "Gen": "test.mal_Mlym"
        }
    },
    "Marathi": {
        "code": "mar_Deva",
        "filename": {
            "Conv": "test (1).mar_Deva",
            "Gen": "test.mar_Deva"
        }
    },
    "Odia": {
        "code": "ory_Orya",
        "filename": {
            "Conv": "test.ory_Orya",
            "Gen": "test.ory_Orya"
        }
    }
}

ENGLISH_FILES = {
    "Conv": "test (1).eng_Latn",
    "Gen": "test.eng_Latn"
}


def read_lines(file_path):
    """Read one sentence per line while preserving UTF-8 text."""
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n\r") for line in f]


for subset in ["Conv", "Gen"]:

    print("\n" + "=" * 80)
    print(f"PROCESSING IN22 - {subset}")
    print("=" * 80)

    subset_output_dir = OUTPUT_DIR / subset
    subset_output_dir.mkdir(parents=True, exist_ok=True)

    # ---------------------------------------------------------
    # Load English
    # ---------------------------------------------------------
    english_path = IN22_DIR / subset / ENGLISH_FILES[subset]

    english_sentences = read_lines(english_path)

    print(f"English sentences: {len(english_sentences)}")

    # ---------------------------------------------------------
    # Process each Indic language
    # ---------------------------------------------------------
    for language, config in LANGUAGES.items():

        print("\n" + "-" * 70)
        print(f"{language}")
        print("-" * 70)

        indic_path = IN22_DIR / subset / config["filename"][subset]

        indic_sentences = read_lines(indic_path)

        print(f"Indic sentences: {len(indic_sentences)}")

        # Check alignment
        if len(english_sentences) != len(indic_sentences):
            raise ValueError(
                f"Length mismatch for {subset} - {language}: "
                f"English={len(english_sentences)}, "
                f"{language}={len(indic_sentences)}"
            )

        # =====================================================
        # 1. English → Indic
        # =====================================================

        print(f"\nGenerating English → {language}...")

        tokenizer.src_lang = "eng_Latn"

        en_to_indic = translate_batch(
            english_sentences,
            config["code"],
            batch_size=16
        )

        forward_df = pd.DataFrame({
            "source": english_sentences,
            "reference": indic_sentences,
            "translation": en_to_indic,
            "source_language": "English",
            "target_language": language,
            "direction": f"eng_Latn→{config['code']}",
            "dataset": "IN22",
            "subset": subset
        })

        forward_file = subset_output_dir / f"en_to_{language.lower()}.csv"

        forward_df.to_csv(
            forward_file,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"Saved: {forward_file}")

        # =====================================================
        # 2. Indic → English
        # =====================================================

        print(f"\nGenerating {language} → English...")

        tokenizer.src_lang = config["code"]

        indic_to_en = translate_batch(
            indic_sentences,
            "eng_Latn",
            batch_size=16
        )

        reverse_df = pd.DataFrame({
            "source": indic_sentences,
            "reference": english_sentences,
            "translation": indic_to_en,
            "source_language": language,
            "target_language": "English",
            "direction": f"{config['code']}→eng_Latn",
            "dataset": "IN22",
            "subset": subset
        })

        reverse_file = subset_output_dir / f"{language.lower()}_to_en.csv"

        reverse_df.to_csv(
            reverse_file,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"Saved: {reverse_file}")

print("\n" + "=" * 80)
print("IN22 BIDIRECTIONAL GENERATION COMPLETE")
print("=" * 80)


PROCESSING IN22 - Conv
English sentences: 1503

----------------------------------------------------------------------
Gujarati
----------------------------------------------------------------------
Indic sentences: 1503

Generating English → Gujarati...


Translating → guj_Gujr:   0%|          | 0/94 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/in22/Conv/en_to_gujarati.csv

Generating Gujarati → English...


Translating → eng_Latn:   0%|          | 0/94 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/in22/Conv/gujarati_to_en.csv

----------------------------------------------------------------------
Hindi
----------------------------------------------------------------------
Indic sentences: 1503

Generating English → Hindi...


...

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/in22/Gen/marathi_to_en.csv

----------------------------------------------------------------------
Odia
----------------------------------------------------------------------
Indic sentences: 1024

Generating English → Odia...


Translating → ory_Orya:   0%|          | 0/64 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/in22/Gen/en_to_odia.csv

Generating Odia → English...


Translating → eng_Latn:   0%|          | 0/64 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/UniMTScore/processed/mt_outputs/in22/Gen/odia_to_en.csv

IN22 BIDIRECTIONAL GENERATION COMPLETE


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

EVAL_DIR = PROJECT_ROOT / "processed" / "evaluation_ready"

EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Evaluation directory:")
print(EVAL_DIR)

Evaluation directory:
/content/drive/MyDrive/UniMTScore/processed/evaluation_ready


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

FLORES_FORWARD = PROJECT_ROOT / "processed" / "mt_outputs" / "flores200"
FLORES_REVERSE = PROJECT_ROOT / "processed" / "mt_outputs" / "flores200_reverse"

IN22_DIR = PROJECT_ROOT / "processed" / "mt_outputs" / "in22"

all_data = []

# ============================================================
# FLORES-200
# ============================================================

print("Loading FLORES-200...")

for file in sorted(FLORES_FORWARD.glob("*.csv")):

    df = pd.read_csv(file)

    # Standardize columns
    if "source_language" not in df.columns:
        # Forward files generated earlier
        pass

    # Add metadata if necessary
    language = file.stem.replace("_mt", "")

    df["dataset"] = "FLORES-200"
    df["subset"] = "dev"

    # Determine language code
    language_codes = {
        "Gujarati": "guj_Gujr",
        "Hindi": "hin_Deva",
        "Kannada": "kan_Knda",
        "Malayalam": "mal_Mlym",
        "Marathi": "mar_Deva",
        "Odia": "ory_Orya"
    }

    df["source_language"] = "English"
    df["target_language"] = language
    df["direction"] = f"eng_Latn→{language_codes[language]}"

    df = df.rename(columns={
        "sentence_eng_Latn": "source",
        "sentence_guj_Gujr": "reference",
        "sentence_hin_Deva": "reference",
        "sentence_kan_Knda": "reference",
        "sentence_mal_Mlym": "reference",
        "sentence_mar_Deva": "reference",
        "sentence_ory_Orya": "reference"
    })

    all_data.append(df)


for file in sorted(FLORES_REVERSE.glob("*.csv")):

    df = pd.read_csv(file)

    df["dataset"] = "FLORES-200"
    df["subset"] = "dev"

    all_data.append(df)


# ============================================================
# IN22
# ============================================================

print("Loading IN22...")

for subset in ["Conv", "Gen"]:

    subset_dir = IN22_DIR / subset

    for file in sorted(subset_dir.glob("*.csv")):

        df = pd.read_csv(file)

        # Metadata already exists in generated files
        all_data.append(df)


print("\nFiles loaded:", len(all_data))

Loading FLORES-200...
Loading IN22...

Files loaded: 36


In [ ]:
combined = pd.concat(
    all_data,
    ignore_index=True
)

print("Combined shape:", combined.shape)

print("\nColumns:")
print(combined.columns.tolist())

print("\nDataset distribution:")
print(combined["dataset"].value_counts())

print("\nDirection distribution:")
print(combined["direction"].value_counts())

Combined shape: (42288, 8)

Columns:
['source', 'reference', 'translation', 'dataset', 'subset', 'source_language', 'target_language', 'direction']

Dataset distribution:
dataset
IN22          30324
FLORES-200    11964
Name: count, dtype: int64

Direction distribution:
direction
eng_Latn→guj_Gujr    3524
eng_Latn→hin_Deva    3524
eng_Latn→kan_Knda    3524
eng_Latn→mal_Mlym    3524
eng_Latn→mar_Deva    3524
eng_Latn→ory_Orya    3524
guj_Gujr→eng_Latn    3524
hin_Deva→eng_Latn    3524
kan_Knda→eng_Latn    3524
mal_Mlym→eng_Latn    3524
mar_Deva→eng_Latn    3524
ory_Orya→eng_Latn    3524
Name: count, dtype: int64


In [ ]:
print("Combined shape:", combined.shape)
print(combined["dataset"].value_counts())
print(combined["direction"].value_counts())

Combined shape: (42288, 8)
dataset
IN22          30324
FLORES-200    11964
Name: count, dtype: int64
direction
eng_Latn→guj_Gujr    3524
eng_Latn→hin_Deva    3524
eng_Latn→kan_Knda    3524
eng_Latn→mal_Mlym    3524
eng_Latn→mar_Deva    3524
eng_Latn→ory_Orya    3524
guj_Gujr→eng_Latn    3524
hin_Deva→eng_Latn    3524
kan_Knda→eng_Latn    3524
mal_Mlym→eng_Latn    3524
mar_Deva→eng_Latn    3524
ory_Orya→eng_Latn    3524
Name: count, dtype: int64


In [ ]:
# ============================================================
# STANDARDIZE UNIFIED EVALUATION DATA
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

EVAL_DIR = PROJECT_ROOT / "processed" / "evaluation_ready"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Required columns
required_columns = [
    "dataset",
    "subset",
    "source",
    "reference",
    "translation",
    "source_language",
    "target_language",
    "direction"
]

# Check columns
missing_columns = [
    col for col in required_columns
    if col not in combined.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are present.")


# Keep only required columns
evaluation_data = combined[required_columns].copy()


# ------------------------------------------------------------
# Remove invalid rows
# ------------------------------------------------------------

before = len(evaluation_data)

evaluation_data = evaluation_data.dropna(
    subset=["source", "reference", "translation"]
)

evaluation_data = evaluation_data[
    (evaluation_data["source"].astype(str).str.strip() != "") &
    (evaluation_data["reference"].astype(str).str.strip() != "") &
    (evaluation_data["translation"].astype(str).str.strip() != "")
]

after = len(evaluation_data)

print(f"\nRows before cleaning: {before}")
print(f"Rows after cleaning:  {after}")
print(f"Rows removed:         {before - after}")


# ------------------------------------------------------------
# Remove exact duplicates
# ------------------------------------------------------------

duplicates = evaluation_data.duplicated().sum()

print("Duplicate rows:", duplicates)

evaluation_data = evaluation_data.drop_duplicates().reset_index(drop=True)


# ------------------------------------------------------------
# Add unique evaluation ID
# ------------------------------------------------------------

evaluation_data.insert(
    0,
    "evaluation_id",
    range(1, len(evaluation_data) + 1)
)


# ------------------------------------------------------------
# Save unified dataset
# ------------------------------------------------------------

output_file = EVAL_DIR / "unified_evaluation_data.csv"

evaluation_data.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(output_file)

print("\nFinal shape:")
print(evaluation_data.shape)

print("\nDataset distribution:")
print(evaluation_data["dataset"].value_counts())

print("\nDirection distribution:")
print(evaluation_data["direction"].value_counts())

All required columns are present.

Rows before cleaning: 42288
Rows after cleaning:  42288
Rows removed:         0
Duplicate rows: 14

Saved:
/content/drive/MyDrive/UniMTScore/processed/evaluation_ready/unified_evaluation_data.csv

Final shape:
(42274, 9)

Dataset distribution:
dataset
IN22          30310
FLORES-200    11964
Name: count, dtype: int64

Direction distribution:
direction
eng_Latn→kan_Knda    3524
mar_Deva→eng_Latn    3524
kan_Knda→eng_Latn    3524
eng_Latn→mar_Deva    3524
hin_Deva→eng_Latn    3523
eng_Latn→hin_Deva    3523
eng_Latn→guj_Gujr    3523
eng_Latn→ory_Orya    3523
ory_Orya→eng_Latn    3523
guj_Gujr→eng_Latn    3523
eng_Latn→mal_Mlym    3520
mal_Mlym→eng_Latn    3520
Name: count, dtype: int64


In [ ]:
print(evaluation_data.head())

print("\nColumn order:")
print(evaluation_data.columns.tolist())

print("\nMissing values:")
print(evaluation_data.isna().sum())

print("\nRows per dataset:")
print(
    evaluation_data
    .groupby(["dataset", "subset"])
    .size()
)

   evaluation_id     dataset subset  \
0              1  FLORES-200    dev   
1              2  FLORES-200    dev   
2              3  FLORES-200    dev   
3              4  FLORES-200    dev   
4              5  FLORES-200    dev   

                                              source  \
0  On Monday, scientists from the Stanford Univer...   
1  Lead researchers say this may bring early dete...   
2  The JAS 39C Gripen crashed onto a runway at ar...   
3  The pilot was identified as Squadron Leader Di...   
4  Local media reports an airport fire vehicle ro...   

                                           reference  \
0  સોમવારે, સ્ટેનફોર્ડ યુનિવર્સિટી સ્કૂલ ઓફ મેડિસ...   
1  અગ્રણી સંશોધનકર્તાઓ કહે છે કે, સમૃદ્ધ દેશોની સ...   
2  JAS 39C ગ્રીપન સવારે 9:30 ના સ્થાનિક સમયે  (02...   
3  પાઇલટની ઓળખ સ્ક્વૉડ્રન લીડર ડિલોક્રિટ પૅટ્ટાવી...   
4  સ્થાનિક મીડિયાના અહેવાલો મુજબ, જવાબ આપતી વખતે ...   

                                         translation source_language  \
0  સોમવારે, સ્ટેનફો

In [ ]:
# Check duplicate rows specifically in IN22 Conv

conv = combined[
    (combined["dataset"] == "IN22") &
    (combined["subset"] == "Conv")
].copy()

print("Original IN22 Conv rows:", len(conv))

duplicate_mask = conv.duplicated(
    subset=[
        "source",
        "reference",
        "translation",
        "source_language",
        "target_language",
        "direction"
    ],
    keep=False
)

duplicates = conv[duplicate_mask].sort_values(
    by=["direction", "source"]
)

print("Rows involved in duplicates:", len(duplicates))

display(duplicates.head(30))

Original IN22 Conv rows: 18036
Rows involved in duplicates: 26


,source,reference,translation,dataset,subset,source_language,target_language,direction
12747,Sure!,ચોક્કસ!,ચોક્કસ!,IN22,Conv,English,Gujarati,eng_Latn→guj_Gujr
12939,Sure!,ચોક્કસ!,ચોક્કસ!,IN22,Conv,English,Gujarati,eng_Latn→guj_Gujr
14424,So cool!,एकदम कूल!,बहुत अच्छा!,IN22,Conv,English,Hindi,eng_Latn→hin_Deva
14444,So cool!,एकदम कूल!,बहुत अच्छा!,IN22,Conv,English,Hindi,eng_Latn→hin_Deva
17430,So cool!,കൂള്‍!,വളരെ നല്ലത്!,IN22,Conv,English,Malayalam,eng_Latn→mal_Mlym
17450,So cool!,കൂള്‍!,വളരെ നല്ലത്!,IN22,Conv,English,Malayalam,eng_Latn→mal_Mlym
17214,What about you?,താനോ?,- നീയെന്താ?,IN22,Conv,English,Malayalam,eng_Latn→mal_Mlym
17220,What about you?,താനോ?,- നീയെന്താ?,IN22,Conv,English,Malayalam,eng_Latn→mal_Mlym
16608,You are welcome.,സന്തോഷം.,സ്വാഗതം.,IN22,Conv,English,Malayalam,eng_Latn→mal_Mlym
17142,You are welcome.,സന്തോഷം.,സ്വാഗതം.,IN22,Conv,English,Malayalam,eng_Latn→mal_Mlym


In [ ]:
# ============================================================
# REBUILD FINAL EVALUATION DATASET
# Preserve all benchmark rows
# ============================================================

evaluation_data = combined[required_columns].copy()

# Remove only rows with missing essential information
before = len(evaluation_data)

evaluation_data = evaluation_data.dropna(
    subset=["source", "reference", "translation"]
)

evaluation_data = evaluation_data[
    (evaluation_data["source"].astype(str).str.strip() != "") &
    (evaluation_data["reference"].astype(str).str.strip() != "") &
    (evaluation_data["translation"].astype(str).str.strip() != "")
].copy()

after = len(evaluation_data)

print("Rows before cleaning:", before)
print("Rows after cleaning:", after)
print("Rows removed:", before - after)

# DO NOT drop duplicates here.
# Repeated benchmark sentences are retained.

evaluation_data = evaluation_data.reset_index(drop=True)

# Add evaluation ID
evaluation_data.insert(
    0,
    "evaluation_id",
    range(1, len(evaluation_data) + 1)
)

# Save
output_file = EVAL_DIR / "unified_evaluation_data.csv"

evaluation_data.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", output_file)
print("Final shape:", evaluation_data.shape)

Rows before cleaning: 42288
Rows after cleaning: 42288
Rows removed: 0

Saved: /content/drive/MyDrive/UniMTScore/processed/evaluation_ready/unified_evaluation_data.csv
Final shape: (42288, 9)


In [ ]:
print(
    evaluation_data
    .groupby(["dataset", "subset"])
    .size()
)

print("\nTotal rows:", len(evaluation_data))

print("\nMissing values:")
print(evaluation_data.isna().sum())

dataset     subset
FLORES-200  dev       11964
IN22        Conv      18036
            Gen       12288
dtype: int64

Total rows: 42288

Missing values:
evaluation_id      0
dataset            0
subset             0
source             0
reference          0
translation        0
source_language    0
target_language    0
direction          0
dtype: int64
